Libraries and Data

In [ ]:
import spacy
import pandas as pd
import re
from pathlib import Path
from spacy.matcher import Matcher

ROOT = Path('../../data/raw')
data = ROOT / 'cases.csv'
metadata = ROOT / 'metadata.csv'


Visualizing

In [6]:
data = pd.read_csv(data)
metadata = pd.read_csv(metadata)

data.head()
metadata.head()


,article_id,authors,case_amount,doi,journal,journal_detail,keywords,license,link,major_mesh_terms,mesh_terms,pmcid,pmid,title,year
0,PMC5137649,"[C E Bailey, M B Fritz, L Webb, N B Merchant, ...",1,10.1308/003588414X13824511649977,Ann R Coll Surg Engl,2014 Jan;96(1):88E-90E.,NaN,CC BY,https://pubmed.ncbi.nlm.nih.gov/24417851/,"[Cysts / diagnosis, Stomach / abnormalities, S...","[Cysts / diagnosis, Stomach / abnormalities, S...",PMC5137649,24417851,Gastric duplication cyst masquerading as a muc...,2014
1,PMC9387390,"[Elias A Chamely, Bryan Hoang, Nadim S Jafri, ...",1,10.4293/CRSLS.2021.00094,CRSLS,2022 Feb 25;9(1):e2021.00094.,"[delayed gastric emptying, endoscopy, gastric ...",CC BY-NC-SA,https://pubmed.ncbi.nlm.nih.gov/36016812/,"[Adenocarcinoma / complications, Gastric Bypas...","[Adenocarcinoma / complications, Gastric Bypas...",PMC9387390,36016812,Palliative Endoscopic Salvage of a Functionall...,2022
2,PMC3437073,[M Y Al-Naami],1,NaN,J Family Community Med,1999 Jan;6(1):45-8.,"[abscess, spleen, tuberculous]",CC BY-NC-SA,https://pubmed.ncbi.nlm.nih.gov/23008596/,[],[Case Reports],PMC3437073,23008596,An unusual presentation of tuberculous splenic...,1999
3,PMC7102447,"[Zeid Nesheiwat, Pinang Shastri, Rohit Vyas, C...",1,10.1155/2020/7842591,Case Rep Cardiol,2020 Jan 11;2020:7842591.,NaN,CC BY,https://pubmed.ncbi.nlm.nih.gov/32257451/,[],[Case Reports],PMC7102447,32257451,A Case of Acute Massive Bioprosthetic Mitral V...,2020
4,PMC6354154,"[Patrícia Alves, Inês Sá, Miguel Brito, Cátia ...",1,10.1155/2019/2537480,Case Rep Obstet Gynecol,2019 Jan 16;2019:2537480.,NaN,CC BY,https://pubmed.ncbi.nlm.nih.gov/30792930/,[],[Case Reports],PMC6354154,30792930,An Early Diagnosis of an Ovarian Steroid Cell ...,2019


inner joining important mesh terms

In [7]:
full_data = pd.merge(data, metadata)
full_data = full_data[['case_text','gender', 'case_id', 'major_mesh_terms', 'mesh_terms']]
full_data.head()

,case_text,gender,case_id,major_mesh_terms,mesh_terms
0,A 44-year-old woman presented with a 3-day his...,Female,PMC5137649_01,"[Cysts / diagnosis, Stomach / abnormalities, S...","[Cysts / diagnosis, Stomach / abnormalities, S..."
1,A 57-year-old man with no significant past med...,Male,PMC9387390_01,"[Adenocarcinoma / complications, Gastric Bypas...","[Adenocarcinoma / complications, Gastric Bypas..."
2,A 55-year-old male presented with a gradually ...,Male,PMC3437073_01,[],[Case Reports]
3,A 65-year-old male with a past medical history...,Male,PMC7102447_01,[],[Case Reports]
4,A 30-year-old nulligravida presented herself i...,Female,PMC6354154_01,[],[Case Reports]


picking the first case_text

In [8]:
text = full_data['case_text'][0]
print(text)

A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparoscopic distal panc

In [15]:
nlp = spacy.load('en_core_web_sm')
processed_text = nlp(text)

for token in processed_text:
    print(f'text: {token.text}, lema: {token.lemma_}, POS: {token.pos_}, lower: {token.lower_}')

text: A, lema: a, POS: DET, lower: a
text: 44, lema: 44, POS: NUM, lower: 44
text: -, lema: -, POS: PUNCT, lower: -
text: year, lema: year, POS: NOUN, lower: year
text: -, lema: -, POS: PUNCT, lower: -
text: old, lema: old, POS: ADJ, lower: old
text: woman, lema: woman, POS: NOUN, lower: woman
text: presented, lema: present, POS: VERB, lower: presented
text: with, lema: with, POS: ADP, lower: with
text: a, lema: a, POS: DET, lower: a
text: 3, lema: 3, POS: NUM, lower: 3
text: -, lema: -, POS: PUNCT, lower: -
text: day, lema: day, POS: NOUN, lower: day
text: history, lema: history, POS: NOUN, lower: history
text: of, lema: of, POS: ADP, lower: of
text: right, lema: right, POS: ADJ, lower: right
text: flank, lema: flank, POS: NOUN, lower: flank
text: and, lema: and, POS: CCONJ, lower: and
text: lower, lema: low, POS: ADJ, lower: lower
text: quadrant, lema: quadrant, POS: ADJ, lower: quadrant
text: abdominal, lema: abdominal, POS: ADJ, lower: abdominal
text: pain, lema: pain, POS: NOUN, l

## Captura de numeros e unidades
utilizando regex

In [24]:
measurement_pattern = re.compile(r'(\d+(?:,\d+)?(?:\.\d+)?)\s*(cm|mm|ng/ml|iu/ml|mg)')

measurements = []
for match in measurement_pattern.finditer(text):
    value = match.group(1)
    unit = match.group(2)

    measurements.append({
        'node_type': 'ExamResult',
        'value': value, 
        'unit': unit, 
        'label_original': match.group(0),
        'label_normalizado': f"{value} {unit}",
        'token_start': -1, # Valores capturados por regex entram como atributos extras
        'token_end': -1,
        'span_start': match.start(),
        'span_end': match.end()
    })

measurements_df = pd.DataFrame(measurements)
print("Resultados de Exames Extraídos:")
display(measurements_df)

Resultados de Exames Extraídos:


,node_type,value,unit,label_original,label_normalizado,token_start,token_end,span_start,span_end
0,ExamResult,6,cm,6cm,6 cm,-1,-1,337,340
1,ExamResult,6,cm,6cm,6 cm,-1,-1,516,519
2,ExamResult,9,cm,9cm,9 cm,-1,-1,522,525
3,ExamResult,"12,476.5",ng/ml,"12,476.5ng/ml","12,476.5 ng/ml",-1,-1,777,790
4,ExamResult,6,iu/ml,6iu/ml,6 iu/ml,-1,-1,837,843
5,ExamResult,9.5,cm,9.5cm,9.5 cm,-1,-1,2070,2075
6,ExamResult,4.5,cm,4.5cm,4.5 cm,-1,-1,2078,2083
7,ExamResult,2.0,cm,2.0cm,2.0 cm,-1,-1,2086,2091


## Matcher baseado em padrões gramaticais
Adjetivo + Substantivo médico ou Substantivos compostos

In [21]:
matcher = Matcher(nlp.vocab)

pattern_clinical = [{"POS": {"IN": ["ADJ", "NOUN"]}}, {"POS": "NOUN"}]
matcher.add("CLINICAL_TERM", [pattern_clinical])

matches = matcher(processed_text)
extracted_entities = []

for match_id, start, end in matches:
    span = processed_text[start:end]
    extracted_entities.append({
        'node_type': 'MedicalConcept',
        'label_original': span.text,
        'label_normalizado': span.lemma_, # Lematização para unificar plural/singular
        'token_start': start,
        'token_end': end,
        'span_start': span.start_char,
        'span_end': span.end_char
    })

nodes_df = pd.DataFrame(extracted_entities).drop_duplicates(subset=['label_normalizado']).reset_index(drop=True)

display(nodes_df.head(15))

,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end
0,MedicalConcept,old woman,old woman,5,7,10,19
1,MedicalConcept,day history,day history,12,14,39,50
2,MedicalConcept,right flank,right flank,15,17,54,65
3,MedicalConcept,abdominal pain,abdominal pain,20,22,85,99
4,MedicalConcept,medication history,medication history,34,36,170,188
5,MedicalConcept,physical examination,physical examination,43,45,229,249
6,MedicalConcept,computed tomography,computed tomography,52,54,300,319
7,MedicalConcept,cystic lesion,cystic lesion,59,61,341,354
8,MedicalConcept,pancreatic echotexture,pancreatic echotexture,86,88,472,494
9,MedicalConcept,internal septations,internal septation,101,103,543,562


## Extração de Arestas Semânticas por Dependência Sintática 

In [22]:

dependency_edges = []
edge_id_counter = 1

for sent in processed_text.sents:
    nodes_in_sent = [row for _, row in nodes_df.iterrows() if sent.start <= row['token_start'] < sent.end]
    
    if len(nodes_in_sent) >= 2:
        for token in sent:
            if token.pos_ == "VERB":
                verbo_lema = token.lemma_.lower()
                
                relacao = "RELATED_TO"
                if verbo_lema in ["present", "have", "experience"]:
                    relacao = "HAS_SYMPTOM"
                elif verbo_lema in ["undergo", "perform", "do", "plan"]:
                    relacao = "UNDERWENT_EXAM"
                elif verbo_lema in ["reveal", "demonstrate", "suggest", "show", "consist"]:
                    relacao = "SUPPORTS"
                elif verbo_lema in ["treat", "resect", "discharge", "perform"]:
                    relacao = "TREATED_BY"

                for i in range(len(nodes_in_sent) - 1):
                    source = nodes_in_sent[i]
                    target = nodes_in_sent[i + 1]
                    
                    dependency_edges.append({
                        'edge_id': f"E_DEP_{edge_id_counter:03d}",
                        'source_label': source['label_normalizado'],
                        'target_label': target['label_normalizado'],
                        'relation': relacao,
                        'trigger_verb': verbo_lema,
                        'sentence': sent.text.strip()
                    })
                    edge_id_counter += 1

edges_dep_df = pd.DataFrame(dependency_edges).drop_duplicates(subset=['source_label', 'target_label', 'relation'])

print(f"Total de arestas por dependência sintática: {len(edges_dep_df)}")
display(edges_dep_df.head(10))

Total de arestas por dependência sintática: 41


,edge_id,source_label,target_label,relation,trigger_verb,sentence
0,E_DEP_001,old woman,day history,HAS_SYMPTOM,present,A 44-year-old woman presented with a 3-day his...
1,E_DEP_002,day history,right flank,HAS_SYMPTOM,present,A 44-year-old woman presented with a 3-day his...
2,E_DEP_003,right flank,abdominal pain,HAS_SYMPTOM,present,A 44-year-old woman presented with a 3-day his...
3,E_DEP_004,old woman,day history,RELATED_TO,associate,A 44-year-old woman presented with a 3-day his...
4,E_DEP_005,day history,right flank,RELATED_TO,associate,A 44-year-old woman presented with a 3-day his...
5,E_DEP_006,right flank,abdominal pain,RELATED_TO,associate,A 44-year-old woman presented with a 3-day his...
6,E_DEP_007,computed tomography,cystic lesion,UNDERWENT_EXAM,undergo,She underwent contrast enhanced computed tomog...
7,E_DEP_008,computed tomography,cystic lesion,RELATED_TO,enhance,She underwent contrast enhanced computed tomog...
8,E_DEP_009,computed tomography,cystic lesion,SUPPORTS,demonstrate,She underwent contrast enhanced computed tomog...
9,E_DEP_010,pancreatic echotexture,internal septation,UNDERWENT_EXAM,undergo,"She subsequently underwent EUS-FNA, which reve..."


## Unir Entidades e Medidas Regex em um Grafo

In [26]:
final_nodes_df = pd.concat([nodes_df, measurements_df], ignore_index=True)

print(f"Total de Nós Unificados: {len(final_nodes_df)}")
display(final_nodes_df.head(10))

# 3. criar arestas por proximidade de caracteres
combined_edges = []
edge_id_counter = 1

for _, ent_row in nodes_df.iterrows():
    for _, meas_row in measurements_df.iterrows():
        distancia = abs(ent_row['span_start'] - meas_row['span_start'])
        if distancia < 40: 
            combined_edges.append({
                'edge_id': f"E_VAL_{edge_id_counter:03d}",
                'source_label': ent_row['label_normalizado'],
                'target_label': meas_row['label_normalizado'],
                'relation': 'HAS_VALUE',
                'character_distance': distancia
            })
            edge_id_counter += 1

final_edges_df = pd.DataFrame(combined_edges).drop_duplicates(subset=['source_label', 'target_label'])

print(f"\nTotal de Arestas de Valores geradas: {len(final_edges_df)}")
display(final_edges_df)

Total de Nós Unificados: 51


,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,MedicalConcept,old woman,old woman,5,7,10,19,NaN,NaN
1,MedicalConcept,day history,day history,12,14,39,50,NaN,NaN
2,MedicalConcept,right flank,right flank,15,17,54,65,NaN,NaN
3,MedicalConcept,abdominal pain,abdominal pain,20,22,85,99,NaN,NaN
4,MedicalConcept,medication history,medication history,34,36,170,188,NaN,NaN
5,MedicalConcept,physical examination,physical examination,43,45,229,249,NaN,NaN
6,MedicalConcept,computed tomography,computed tomography,52,54,300,319,NaN,NaN
7,MedicalConcept,cystic lesion,cystic lesion,59,61,341,354,NaN,NaN
8,MedicalConcept,pancreatic echotexture,pancreatic echotexture,86,88,472,494,NaN,NaN
9,MedicalConcept,internal septations,internal septation,101,103,543,562,NaN,NaN



Total de Arestas de Valores geradas: 13


,edge_id,source_label,target_label,relation,character_distance
0,E_VAL_001,computed tomography,6 cm,HAS_VALUE,37
1,E_VAL_002,cystic lesion,6 cm,HAS_VALUE,4
2,E_VAL_003,internal septation,6 cm,HAS_VALUE,27
3,E_VAL_004,internal septation,9 cm,HAS_VALUE,21
4,E_VAL_005,carbohydrate antigen,"12,476.5 ng/ml",HAS_VALUE,20
5,E_VAL_006,final pathology,9.5 cm,HAS_VALUE,27
6,E_VAL_007,final pathology,4.5 cm,HAS_VALUE,35
7,E_VAL_008,cm cyst,9.5 cm,HAS_VALUE,19
8,E_VAL_009,cm cyst,4.5 cm,HAS_VALUE,11
9,E_VAL_010,cm cyst,2.0 cm,HAS_VALUE,3


## UNir todos os nós e arestas

In [ ]:
#Padroniza as arestas de dependência sintática e de valores numéricos
edges_dep_clean = edges_dep_df[['edge_id', 'source_label', 'target_label', 'relation']].copy()
edges_val_clean = final_edges_df[['edge_id', 'source_label', 'target_label', 'relation']].copy()

final_graph_edges_df = pd.concat([edges_dep_clean, edges_val_clean], ignore_index=True)
final_graph_edges_df = final_graph_edges_df.drop_duplicates(subset=['source_label', 'target_label', 'relation']).reset_index(drop=True)

# Ajusta o ID das arestas sequencialmente
final_graph_edges_df['edge_id'] = [f"E_{i+1:03d}" for i in range(len(final_graph_edges_df))]

print("=== GRAFO DE CONHECIMENTO CONSOLIDADO ===")
print(f"Total de Nós Finais: {len(final_nodes_df)}")
print(f"Total de Arestas Finais: {len(final_graph_edges_df)}")

display(final_graph_edges_df.head(15))

=== GRAFO DE CONHECIMENTO CONSOLIDADO ===
Total de Nós Finais: 51
Total de Arestas Finais: 54


,edge_id,source_label,target_label,relation
0,E_001,old woman,day history,HAS_SYMPTOM
1,E_002,day history,right flank,HAS_SYMPTOM
2,E_003,right flank,abdominal pain,HAS_SYMPTOM
3,E_004,old woman,day history,RELATED_TO
4,E_005,day history,right flank,RELATED_TO
5,E_006,right flank,abdominal pain,RELATED_TO
6,E_007,computed tomography,cystic lesion,UNDERWENT_EXAM
7,E_008,computed tomography,cystic lesion,RELATED_TO
8,E_009,computed tomography,cystic lesion,SUPPORTS
9,E_010,pancreatic echotexture,internal septation,UNDERWENT_EXAM
